# 01. 데이터 이해 (Data Understanding)

## 1. 프로젝트 및 데이터 이해

### 프로젝트 목적

본 프로젝트는 서울시 상권분석서비스 데이터를 활용하여 **황학동 상권의 변화**를 분석한다.

황학동이 단순히 쇠퇴하는 전통시장 상권인지, 아니면 기존 전통시장 상권과 신규 소비 상권(예: 카페, 신흥 업종 등)이 동시에 존재하는 **'전환 중인 상권'** 인지를 데이터에 근거하여 검증하는 것이 핵심 목적이다.

### 분석 기간

- **2021년 1분기 (2021Q1) ~ 2025년 4분기 (2025Q4)**

### 사용 데이터 (4종)

| 데이터 | 원본명 | 설명 |
|---|---|---|
| `sales` | 서울시 상권분석서비스 추정매출-상권 | 상권 × 업종 단위 분기별 추정 매출 |
| `stores` | 서울시 상권분석서비스 점포-상권 | 상권 × 업종 단위 분기별 점포 수/개폐업 현황 |
| `population` | 서울시 상권분석서비스 길단위인구-상권 | 상권 단위 분기별 유동인구 |
| `commercial_area` | 서울시 상권분석서비스 영역-상권 GIS | 상권 경계 폴리곤(Shapefile) |

### 이 노트북의 목적

이 노트북은 **EDA(탐색적 데이터 분석)나 전처리를 위한 것이 아니다.**

목적은 오직 다음 두 가지다.

1. **원본 데이터의 구조를 있는 그대로 확인**한다 (파일 수, 컬럼, 인코딩, 기간 커버리지 등).
2. **데이터 간 결합(join) 가능성을 검증**한다 (공통 key, 관측 단위/grain 등).

결측치 처리, 이상치 처리, 컬럼 삭제/변경, feature engineering, merge, 그래프 기반 EDA, 모델링, `data/processed` 생성 등은 **이후 노트북에서 다룬다.** 또한 `data/raw`의 원본 파일은 이 노트북에서 어떠한 방식으로도 수정하지 않는다.

## 2. 라이브러리 및 경로 설정

이 노트북은 `notebooks/` 폴더에 위치하므로, `data/raw`는 상위 폴더 기준 상대경로(`../data/raw`)로 접근한다. 절대경로는 사용하지 않는다.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("../data/raw")
SALES_DIR = DATA_DIR / "sales"
STORES_DIR = DATA_DIR / "stores"
POPULATION_DIR = DATA_DIR / "population"
COMMERCIAL_AREA_DIR = DATA_DIR / "commercial_area"

for name, d in [
    ("SALES_DIR", SALES_DIR),
    ("STORES_DIR", STORES_DIR),
    ("POPULATION_DIR", POPULATION_DIR),
    ("COMMERCIAL_AREA_DIR", COMMERCIAL_AREA_DIR),
]:
    print(f"{name:22s} {d} -> exists: {d.exists()}")

SALES_DIR              ..\data\raw\sales -> exists: True
STORES_DIR             ..\data\raw\stores -> exists: True
POPULATION_DIR         ..\data\raw\population -> exists: True
COMMERCIAL_AREA_DIR    ..\data\raw\commercial_area -> exists: True


## 3. 파일 존재 여부 및 파일 목록 확인

예상되는 파일 수(sales 5개, stores 5개, population 20개)를 미리 가정하지 않고, 실제 폴더를 스캔하여 몇 개의 파일이 있는지 확인한다.

In [2]:
sales_files = sorted(SALES_DIR.glob("*.csv"))
stores_files = sorted(STORES_DIR.glob("*.csv"))
population_files = sorted(POPULATION_DIR.glob("*.csv"))
commercial_area_files = sorted(COMMERCIAL_AREA_DIR.glob("*"))

print(f"[sales] 파일 수: {len(sales_files)}")
for f in sales_files:
    print(" -", f.name)

print(f"\n[stores] 파일 수: {len(stores_files)}")
for f in stores_files:
    print(" -", f.name)

print(f"\n[population] 파일 수: {len(population_files)}")
for f in population_files:
    print(" -", f.name)

print(f"\n[commercial_area] 파일 목록 (총 {len(commercial_area_files)}개)")
for f in commercial_area_files:
    print(" -", f.name)

[sales] 파일 수: 5
 - sales_2021.csv
 - sales_2022.csv
 - sales_2023.csv
 - sales_2024.csv
 - sales_2025.csv

[stores] 파일 수: 5
 - stores_2021.csv
 - stores_2022.csv
 - stores_2023.csv
 - stores_2024.csv
 - stores_2025.csv

[population] 파일 수: 20
 - population_20211.csv
 - population_20212.csv
 - population_20213.csv
 - population_20214.csv
 - population_20221.csv
 - population_20222.csv
 - population_20223.csv
 - population_20224.csv
 - population_20231.csv
 - population_20232.csv
 - population_20233.csv
 - population_20234.csv
 - population_20241.csv
 - population_20242.csv
 - population_20243.csv
 - population_20244.csv
 - population_20251.csv
 - population_20252.csv
 - population_20253.csv
 - population_20254.csv

[commercial_area] 파일 목록 (총 5개)
 - commercial_area.cpg
 - commercial_area.dbf
 - commercial_area.prj
 - commercial_area.shp
 - commercial_area.shx


## 4. CSV 인코딩 확인 및 샘플 로드

서울시 공공데이터 CSV는 UTF-8이 아닌 `cp949`(EUC-KR 계열)로 배포되는 경우가 많다. 컬럼명이 한글이므로, 먼저 UTF-8 계열로 읽어보고 실패하면 `cp949`로 재시도하여 실제로 한글이 정상적으로 읽히는 인코딩을 확인한다.

In [3]:
sample_path = sales_files[0]

# 1) UTF-8 계열 시도
try:
    _test = pd.read_csv(sample_path, encoding="utf-8", nrows=5)
    print("UTF-8: 정상적으로 읽힘")
    print(_test.columns[:3].tolist())
except UnicodeDecodeError as e:
    print("UTF-8: 디코딩 실패 ->", e)

# 2) cp949 시도
try:
    _test = pd.read_csv(sample_path, encoding="cp949", nrows=5)
    print("\ncp949: 정상적으로 읽힘")
    print("컬럼 예시:", _test.columns[:3].tolist())
except UnicodeDecodeError as e:
    print("cp949: 디코딩 실패 ->", e)

UTF-8: 디코딩 실패 -> 'utf-8' codec can't decode byte 0xb1 in position 0: invalid start byte

cp949: 정상적으로 읽힘
컬럼 예시: ['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명']


In [4]:
def read_csv_kr(path, **kwargs):
    """UTF-8 계열을 우선 시도하고, 실패하면 cp949로 재시도해서 읽는다."""
    for enc in ("utf-8", "utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"지원하는 인코딩으로 읽을 수 없습니다: {path}")


def show_summary(name, df):
    """데이터프레임의 구조(shape, head, info, 컬럼명)를 보기 좋게 출력한다."""
    print(f"\n{'=' * 15} {name} {'=' * 15}")
    print("shape:", df.shape)
    display(df.head(3))
    print("\n[info]")
    df.info()
    print(f"\n[컬럼명 목록] (총 {len(df.columns)}개)")
    cols = df.columns.tolist()
    for i in range(0, len(cols), 5):
        print(" ", ", ".join(cols[i:i + 5]))

In [5]:
sample_targets = {
    "sales_2021": SALES_DIR / "sales_2021.csv",
    "stores_2021": STORES_DIR / "stores_2021.csv",
    "population_20211": POPULATION_DIR / "population_20211.csv",
}

samples = {}
for name, path in sample_targets.items():
    samples[name] = read_csv_kr(path)

for name, df in samples.items():
    show_summary(name, df)


=============== sales_2021 ===============
shape: (89150, 55)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,월요일_매출_금액,화요일_매출_금액,수요일_매출_금액,목요일_매출_금액,금요일_매출_금액,토요일_매출_금액,일요일_매출_금액,시간대_00~06_매출_금액,시간대_06~11_매출_금액,시간대_11~14_매출_금액,시간대_14~17_매출_금액,시간대_17~21_매출_금액,시간대_21~24_매출_금액,남성_매출_금액,여성_매출_금액,연령대_10_매출_금액,연령대_20_매출_금액,연령대_30_매출_금액,연령대_40_매출_금액,연령대_50_매출_금액,연령대_60_이상_매출_금액,주중_매출_건수,주말_매출_건수,월요일_매출_건수,화요일_매출_건수,수요일_매출_건수,목요일_매출_건수,금요일_매출_건수,토요일_매출_건수,일요일_매출_건수,시간대_건수~06_매출_건수,시간대_건수~11_매출_건수,시간대_건수~14_매출_건수,시간대_건수~17_매출_건수,시간대_건수~21_매출_건수,시간대_건수~24_매출_건수,남성_매출_건수,여성_매출_건수,연령대_10_매출_건수,연령대_20_매출_건수,연령대_30_매출_건수,연령대_40_매출_건수,연령대_50_매출_건수,연령대_60_이상_매출_건수
0,20211,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,382839289,13701,235308851,147530438,35385544,53566081,53687402,47097897,45571927,70698283,76832155,0,2699877,138275545,77483113,157839930,6540824,218578196,102157686,888451,12720922,26575259,59039301,111451438,110060512,9487,4214,1686,2059,2170,1793,1779,2286,1928,0,128,6211,2701,4520,141,8000,4228,102,826,1174,2467,4049,3615
1,20211,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,122957138,12039,61070094,61887044,8611528,15264108,10629354,15464848,11100256,32014369,29872675,11373269,37723773,43587908,26677599,3594589,0,64010667,47678846,251721,4661073,10112557,25115187,48256159,23292816,6435,5604,1120,1319,1463,1164,1369,2809,2795,289,4245,5060,1893,552,0,6679,4293,52,926,1222,2698,3404,2670
2,20211,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,19012827,264,14322928,4689899,1355810,4118094,5868849,726356,2253819,1217565,3472334,0,0,0,460616,9640869,8911342,11667789,3401471,0,0,611202,575770,8576819,5305469,174,90,23,28,52,27,44,37,53,0,0,0,18,188,58,182,45,0,0,9,18,102,97



[info]
<class 'pandas.DataFrame'>
RangeIndex: 89150 entries, 0 to 89149
Data columns (total 55 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   기준_년분기_코드        89150 non-null  int64
 1   상권_구분_코드         89150 non-null  str  
 2   상권_구분_코드_명       89150 non-null  str  
 3   상권_코드            89150 non-null  int64
 4   상권_코드_명          89150 non-null  str  
 5   서비스_업종_코드        89150 non-null  str  
 6   서비스_업종_코드_명      89150 non-null  str  
 7   당월_매출_금액         89150 non-null  int64
 8   당월_매출_건수         89150 non-null  int64
 9   주중_매출_금액         89150 non-null  int64
 10  주말_매출_금액         89150 non-null  int64
 11  월요일_매출_금액        89150 non-null  int64
 12  화요일_매출_금액        89150 non-null  int64
 13  수요일_매출_금액        89150 non-null  int64
 14  목요일_매출_금액        89150 non-null  int64
 15  금요일_매출_금액        89150 non-null  int64
 16  토요일_매출_금액        89150 non-null  int64
 17  일요일_매출_금액        89150 non-null  int64
 18  시간대_00~06

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20211,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,12,0,0,0,0,0
1,20211,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,0,0
2,20211,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,3,4,0,0,0,0,1



[info]
<class 'pandas.DataFrame'>
RangeIndex: 303880 entries, 0 to 303879
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   기준_년분기_코드    303880 non-null  int64
 1   상권_구분_코드     303880 non-null  str  
 2   상권_구분_코드_명   303880 non-null  str  
 3   상권_코드        303880 non-null  int64
 4   상권_코드_명      303880 non-null  str  
 5   서비스_업종_코드    303880 non-null  str  
 6   서비스_업종_코드_명  303880 non-null  str  
 7   점포_수         303880 non-null  int64
 8   유사_업종_점포_수   303880 non-null  int64
 9   개업_율         303880 non-null  int64
 10  개업_점포_수      303880 non-null  int64
 11  폐업_률         303880 non-null  int64
 12  폐업_점포_수      303880 non-null  int64
 13  프랜차이즈_점포_수   303880 non-null  int64
dtypes: int64(9), str(5)
memory usage: 48.2 MB

[컬럼명 목록] (총 14개)
  기준_년분기_코드, 상권_구분_코드, 상권_구분_코드_명, 상권_코드, 상권_코드_명
  서비스_업종_코드, 서비스_업종_코드_명, 점포_수, 유사_업종_점포_수, 개업_율
  개업_점포_수, 폐업_률, 폐업_점포_수, 프랜차이즈_점포_수

=============== population_20211 =

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,총_유동인구_수,남성_유동인구_수,여성_유동인구_수,연령대_10_유동인구_수,연령대_20_유동인구_수,연령대_30_유동인구_수,연령대_40_유동인구_수,연령대_50_유동인구_수,연령대_60_이상_유동인구_수,시간대_00_06_유동인구_수,시간대_06_11_유동인구_수,시간대_11_14_유동인구_수,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,월요일_유동인구_수,화요일_유동인구_수,수요일_유동인구_수,목요일_유동인구_수,금요일_유동인구_수,토요일_유동인구_수,일요일_유동인구_수
0,20211,R,전통시장,3130093,동서시장,55869,28962,26906,2383,4236,5347,7525,10818,25561,7701,11527,10785,11447,9583,4824,8392,8465,8576,8219,7926,7760,6530
1,20211,A,골목상권,3111004,삼전역 1번,2228673,1035991,1192681,346754,341951,450172,381968,285522,422307,627736,465543,245684,239313,352222,298173,319258,318087,318491,317386,314453,321281,319715
2,20211,A,골목상권,3111005,삼전역 3번,1297239,597924,699315,194503,185054,279049,213435,181337,243861,430080,279578,124080,110097,171886,181519,186155,182724,182943,182740,182483,186612,193581



[info]
<class 'pandas.DataFrame'>
RangeIndex: 1650 entries, 0 to 1649
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   기준_년분기_코드         1650 non-null   int64
 1   상권_구분_코드          1650 non-null   str  
 2   상권_구분_코드_명        1650 non-null   str  
 3   상권_코드             1650 non-null   int64
 4   상권_코드_명           1650 non-null   str  
 5   총_유동인구_수          1650 non-null   int64
 6   남성_유동인구_수         1650 non-null   int64
 7   여성_유동인구_수         1650 non-null   int64
 8   연령대_10_유동인구_수     1650 non-null   int64
 9   연령대_20_유동인구_수     1650 non-null   int64
 10  연령대_30_유동인구_수     1650 non-null   int64
 11  연령대_40_유동인구_수     1650 non-null   int64
 12  연령대_50_유동인구_수     1650 non-null   int64
 13  연령대_60_이상_유동인구_수  1650 non-null   int64
 14  시간대_00_06_유동인구_수  1650 non-null   int64
 15  시간대_06_11_유동인구_수  1650 non-null   int64
 16  시간대_11_14_유동인구_수  1650 non-null   int64
 17  시간대_14_17_유동인구_수  1650 non-null   in

## 5. 연도/분기별 파일 구조 일치 여부 확인

여러 연도의 파일을 이후에 하나로 합쳐서 사용하려면, 연도별 파일의 컬럼 구성(개수 + 컬럼명 + 순서)이 모두 같아야 한다. 단순히 컬럼 개수만 비교하면 순서 변경이나 컬럼명 오탈자를 놓칠 수 있으므로, 컬럼명 리스트 자체를 비교한다. 헤더만 필요하므로 `nrows=0`으로 빠르게 읽는다.

In [6]:
def get_columns(path):
    return read_csv_kr(path, nrows=0).columns.tolist()


def check_column_consistency(files, label):
    """첫 파일의 컬럼(이름+순서)을 기준으로 나머지 파일들을 비교하고 차이를 출력한다."""
    base_cols = get_columns(files[0])
    print(f"[{label}] 기준 파일: {files[0].name} (컬럼 {len(base_cols)}개)")

    all_same = True
    for f in files[1:]:
        cols = get_columns(f)
        if cols == base_cols:
            print(f"  - {f.name}: 동일 (컬럼 {len(cols)}개)")
            continue

        all_same = False
        print(f"  - {f.name}: 차이 발견 (컬럼 {len(cols)}개)")
        only_in_base = [c for c in base_cols if c not in cols]
        only_in_this = [c for c in cols if c not in base_cols]
        if only_in_base:
            print(f"      기준 파일에만 있는 컬럼: {only_in_base}")
        if only_in_this:
            print(f"      이 파일에만 있는 컬럼: {only_in_this}")
        if not only_in_base and not only_in_this and cols != base_cols:
            print("      컬럼 구성은 동일하지만 순서가 다름")

    print(f"  -> {label}: {'전체 파일 컬럼 구조 동일' if all_same else '컬럼 구조 불일치 존재'}\n")
    return base_cols


sales_base_cols = check_column_consistency(sales_files, "sales")
stores_base_cols = check_column_consistency(stores_files, "stores")
population_base_cols = check_column_consistency(population_files, "population")

[sales] 기준 파일: sales_2021.csv (컬럼 55개)
  - sales_2022.csv: 동일 (컬럼 55개)
  - sales_2023.csv: 동일 (컬럼 55개)
  - sales_2024.csv: 동일 (컬럼 55개)
  - sales_2025.csv: 동일 (컬럼 55개)
  -> sales: 전체 파일 컬럼 구조 동일

[stores] 기준 파일: stores_2021.csv (컬럼 14개)
  - stores_2022.csv: 동일 (컬럼 14개)
  - stores_2023.csv: 동일 (컬럼 14개)
  - stores_2024.csv: 동일 (컬럼 14개)
  - stores_2025.csv: 차이 발견 (컬럼 14개)
      기준 파일에만 있는 컬럼: ['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '점포_수', '유사_업종_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수', '프랜차이즈_점포_수']
      이 파일에만 있는 컬럼: ['stdr_yyqu_cd', 'trdar_se_cd', 'trdar_se_cd_nm', 'trdar_cd', 'trdar_cd_nm', 'svc_induty_cd', 'svc_induty_cd_nm', 'stor_co', 'similr_induty_stor_co', 'opbiz_rt', 'opbiz_stor_co', 'clsbiz_rt', 'clsbiz_stor_co', 'frc_stor_co']
  -> stores: 컬럼 구조 불일치 존재

[population] 기준 파일: population_20211.csv (컬럼 27개)
  - population_20212.csv: 동일 (컬럼 27개)
  - population_20213.csv: 동일 (컬럼 27개)
  - population_20214.csv: 동일 (컬럼 27개)
  - popula

  - population_20244.csv: 동일 (컬럼 27개)
  - population_20251.csv: 동일 (컬럼 27개)
  - population_20252.csv: 동일 (컬럼 27개)


  - population_20253.csv: 동일 (컬럼 27개)
  - population_20254.csv: 동일 (컬럼 27개)
  -> population: 전체 파일 컬럼 구조 동일



## 6. 기준년분기 코드 검증

먼저 각 데이터의 실제 컬럼명 목록에서 "기준"이라는 단어가 포함된 컬럼을 찾아, 기준년분기 역할을 하는 컬럼명을 추측이 아닌 확인을 통해 특정한다.

In [7]:
print("sales 컬럼 중 '기준' 포함:", [c for c in sales_base_cols if "기준" in c])
print("stores 컬럼 중 '기준' 포함:", [c for c in stores_base_cols if "기준" in c])
print("population 컬럼 중 '기준' 포함:", [c for c in population_base_cols if "기준" in c])

sales 컬럼 중 '기준' 포함: ['기준_년분기_코드']
stores 컬럼 중 '기준' 포함: ['기준_년분기_코드']
population 컬럼 중 '기준' 포함: ['기준_년분기_코드']


In [8]:
# 위에서 확인된 실제 컬럼명을 기준년분기 컬럼으로 사용한다 (하드코딩 추측 아님).
sales_quarter_candidates = [c for c in sales_base_cols if "기준" in c]
stores_quarter_candidates = [c for c in stores_base_cols if "기준" in c]
population_quarter_candidates = [c for c in population_base_cols if "기준" in c]

assert sales_quarter_candidates == stores_quarter_candidates == population_quarter_candidates, \
    "세 데이터의 기준년분기 컬럼명이 서로 다릅니다. 위 출력 결과를 다시 확인하세요."

QUARTER_COL = sales_quarter_candidates[0]
print("사용할 기준년분기 컬럼명:", QUARTER_COL)


def resolve_column(path, base_cols, target_col):
    """target_col이 이 파일의 실제 헤더에 없으면, 기준 컬럼 목록(base_cols) 내 위치를 이용해
    같은 위치의 컬럼명을 대신 찾는다. (예: 일부 연도 파일이 한글 대신 영문 코드 컬럼명을 쓰는 경우)"""
    file_cols = get_columns(path)
    if target_col in file_cols:
        return target_col
    if len(file_cols) == len(base_cols) and target_col in base_cols:
        alt = file_cols[base_cols.index(target_col)]
        print(f"    [참고] {path.name}: '{target_col}' 컬럼명이 없어 동일 위치의 '{alt}'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)")
        return alt
    return None


def unique_quarters(path, base_cols):
    col = resolve_column(path, base_cols, QUARTER_COL)
    if col is None:
        print(f"    [경고] {path.name}: 기준년분기에 해당하는 컬럼을 찾지 못해 건너뜀")
        return None
    return sorted(read_csv_kr(path, usecols=[col])[col].unique().tolist())


print("\n[sales] 연도별 파일에 포함된 분기")
for f in sales_files:
    print(f"  - {f.name}: {unique_quarters(f, sales_base_cols)}")

print("\n[stores] 연도별 파일에 포함된 분기")
for f in stores_files:
    print(f"  - {f.name}: {unique_quarters(f, stores_base_cols)}")

사용할 기준년분기 컬럼명: 기준_년분기_코드

[sales] 연도별 파일에 포함된 분기


  - sales_2021.csv: [20211, 20212, 20213, 20214]


  - sales_2022.csv: [20221, 20222, 20223, 20224]


  - sales_2023.csv: [20231, 20232, 20233, 20234]


  - sales_2024.csv: [20241, 20242, 20243, 20244]


  - sales_2025.csv: [20251, 20252, 20253, 20254]

[stores] 연도별 파일에 포함된 분기
  - stores_2021.csv: [20211, 20212, 20213, 20214]


  - stores_2022.csv: [20221, 20222, 20223, 20224]
  - stores_2023.csv: [20231, 20232, 20233, 20234]


  - stores_2024.csv: [20241, 20242, 20243, 20244]
    [참고] stores_2025.csv: '기준_년분기_코드' 컬럼명이 없어 동일 위치의 'stdr_yyqu_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
  - stores_2025.csv: [20251, 20252, 20253, 20254]


In [9]:
# population은 파일명 자체에 분기가 표시되므로(population_20211.csv 등),
# 파일명이 나타내는 분기와 데이터 내부의 기준년분기 코드가 일치하는지 확인한다.
print("[population] 파일명 분기 vs 데이터 내부 기준년분기 코드 일치 확인")
mismatched_files = []
for f in population_files:
    expected = f.stem.replace("population_", "")
    actual = unique_quarters(f, population_base_cols)
    is_match = (actual is not None) and (len(actual) == 1) and (str(actual[0]) == expected)
    status = "OK" if is_match else "확인 필요"
    print(f"  - {f.name}: 파일명 기준={expected}, 데이터 내부 값={actual} -> {status}")
    if not is_match:
        mismatched_files.append(f.name)

print("\n불일치 파일:", mismatched_files if mismatched_files else "없음")

[population] 파일명 분기 vs 데이터 내부 기준년분기 코드 일치 확인
  - population_20211.csv: 파일명 기준=20211, 데이터 내부 값=[20211] -> OK
  - population_20212.csv: 파일명 기준=20212, 데이터 내부 값=[20212] -> OK
  - population_20213.csv: 파일명 기준=20213, 데이터 내부 값=[20213] -> OK
  - population_20214.csv: 파일명 기준=20214, 데이터 내부 값=[20214] -> OK
  - population_20221.csv: 파일명 기준=20221, 데이터 내부 값=[20221] -> OK
  - population_20222.csv: 파일명 기준=20222, 데이터 내부 값=[20222] -> OK
  - population_20223.csv: 파일명 기준=20223, 데이터 내부 값=[20223] -> OK


  - population_20224.csv: 파일명 기준=20224, 데이터 내부 값=[20224] -> OK
  - population_20231.csv: 파일명 기준=20231, 데이터 내부 값=[20231] -> OK
  - population_20232.csv: 파일명 기준=20232, 데이터 내부 값=[20232] -> OK
  - population_20233.csv: 파일명 기준=20233, 데이터 내부 값=[20233] -> OK
  - population_20234.csv: 파일명 기준=20234, 데이터 내부 값=[20234] -> OK
  - population_20241.csv: 파일명 기준=20241, 데이터 내부 값=[20241] -> OK
  - population_20242.csv: 파일명 기준=20242, 데이터 내부 값=[20242] -> OK


  - population_20243.csv: 파일명 기준=20243, 데이터 내부 값=[20243] -> OK
  - population_20244.csv: 파일명 기준=20244, 데이터 내부 값=[20244] -> OK
  - population_20251.csv: 파일명 기준=20251, 데이터 내부 값=[20251] -> OK
  - population_20252.csv: 파일명 기준=20252, 데이터 내부 값=[20252] -> OK
  - population_20253.csv: 파일명 기준=20253, 데이터 내부 값=[20253] -> OK
  - population_20254.csv: 파일명 기준=20254, 데이터 내부 값=[20254] -> OK

불일치 파일: 없음


In [10]:
# 최종적으로 20211 ~ 20254 범위가 실제로 모두 포함되는지 확인한다.
all_sales_quarters = sorted({q for f in sales_files for q in (unique_quarters(f, sales_base_cols) or [])})
all_stores_quarters = sorted({q for f in stores_files for q in (unique_quarters(f, stores_base_cols) or [])})
all_population_quarters = sorted({q for f in population_files for q in (unique_quarters(f, population_base_cols) or [])})

expected_quarters = [int(f"{y}{q}") for y in range(2021, 2026) for q in range(1, 5)]

print("sales 분기 목록     :", all_sales_quarters)
print("stores 분기 목록    :", all_stores_quarters)
print("population 분기 목록:", all_population_quarters)
print("\n기대되는 20211~20254 전체 분기:", expected_quarters)

print("\nsales 누락 분기     :", sorted(set(expected_quarters) - set(all_sales_quarters)))
print("stores 누락 분기    :", sorted(set(expected_quarters) - set(all_stores_quarters)))
print("population 누락 분기:", sorted(set(expected_quarters) - set(all_population_quarters)))

    [참고] stores_2025.csv: '기준_년분기_코드' 컬럼명이 없어 동일 위치의 'stdr_yyqu_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)


sales 분기 목록     : [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244, 20251, 20252, 20253, 20254]
stores 분기 목록    : [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244, 20251, 20252, 20253, 20254]
population 분기 목록: [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244, 20251, 20252, 20253, 20254]

기대되는 20211~20254 전체 분기: [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244, 20251, 20252, 20253, 20254]

sales 누락 분기     : []
stores 누락 분기    : []
population 누락 분기: []


## 7. 상권 관련 핵심 컬럼 확인

앞서 확인한 실제 컬럼명 목록(`sales_base_cols`, `stores_base_cols`, `population_base_cols`)을 대상으로 상권 코드/상권명/상권 구분/서비스 업종/기준년분기 역할을 하는 컬럼이 실제로 존재하는지 문자열 포함 여부로 확인한다. 컬럼명을 임의로 가정하지 않는다.

In [11]:
# (역할, 포함되어야 할 키워드들, 제외되어야 할 키워드들) 형태로 정의하고,
# 실제 컬럼명 목록에서 조건에 맞는 컬럼을 찾는다.
role_rules = [
    ("상권 코드", ["상권", "코드"], ["구분", "명"]),
    ("상권명", ["상권", "명"], ["구분"]),
    ("상권 구분", ["상권", "구분"], []),
    ("서비스 업종 코드", ["서비스", "업종", "코드"], ["명"]),
    ("서비스 업종명", ["서비스", "업종", "명"], []),
    ("기준년분기", ["기준"], []),
]

datasets_cols = {
    "sales": sales_base_cols,
    "stores": stores_base_cols,
    "population": population_base_cols,
}

rows = []
for role, include, exclude in role_rules:
    row = {"역할": role}
    for ds_name, cols in datasets_cols.items():
        matches = [
            c for c in cols
            if all(k in c for k in include) and not any(k in c for k in exclude)
        ]
        row[ds_name] = ", ".join(matches) if matches else "없음"
    rows.append(row)

key_columns_df = pd.DataFrame(rows)
key_columns_df

,역할,sales,stores,population
0,상권 코드,상권_코드,상권_코드,상권_코드
1,상권명,상권_코드_명,상권_코드_명,상권_코드_명
2,상권 구분,"상권_구분_코드, 상권_구분_코드_명","상권_구분_코드, 상권_구분_코드_명","상권_구분_코드, 상권_구분_코드_명"
3,서비스 업종 코드,서비스_업종_코드,서비스_업종_코드,없음
4,서비스 업종명,서비스_업종_코드_명,서비스_업종_코드_명,없음
5,기준년분기,기준_년분기_코드,기준_년분기_코드,기준_년분기_코드


## 8. 황학 관련 상권 탐색

`sales`, `stores`, `population` 각각에서 상권명(`상권_코드_명`)에 '황학'이 포함된 상권을 검색하고, 상권 코드/상권명/상권 구분을 중복 제거하여 정리한다. 이후 세 데이터에서 동일한 상권 코드가 발견되는지 비교한다.

> **주의**: 상권명에 '황학'이라는 문자열이 없더라도, 향후 GIS 경계 데이터를 통해 황학동 행정/계획구역에 포함되는 상권이 있을 수 있다. 따라서 이 단계의 결과만으로 최종 분석 대상 상권 범위를 확정하지 않는다. 이는 어디까지나 데이터 구조 탐색 목적의 1차 스크리닝이다.

In [12]:
# 섹션 7에서 확인된 실제 컬럼명을 그대로 사용한다.
AREA_CODE_COL = [c for c in sales_base_cols if all(k in c for k in ["상권", "코드"]) and not any(k in c for k in ["구분", "명"])][0]
AREA_NAME_COL = [c for c in sales_base_cols if all(k in c for k in ["상권", "명"]) and "구분" not in c][0]
AREA_TYPE_COL = [c for c in sales_base_cols if all(k in c for k in ["상권", "구분"]) and "명" not in c][0]
AREA_TYPE_NAME_COL = [c for c in sales_base_cols if all(k in c for k in ["상권", "구분", "명"])][0]

print("상권 코드 컬럼      :", AREA_CODE_COL)
print("상권명 컬럼         :", AREA_NAME_COL)
print("상권 구분 코드 컬럼 :", AREA_TYPE_COL)
print("상권 구분명 컬럼    :", AREA_TYPE_NAME_COL)

AREA_COLS = [AREA_CODE_COL, AREA_NAME_COL, AREA_TYPE_COL, AREA_TYPE_NAME_COL]


def read_area_cols(path, base_cols):
    """AREA_COLS를 이 파일의 실제 헤더에서 찾아 읽고, 표준 컬럼명으로 통일해서 반환한다.
    (stores_2025.csv처럼 컬럼명이 영문 코드 체계인 파일도 동일하게 처리하기 위한 헬퍼)"""
    resolved = [resolve_column(path, base_cols, c) for c in AREA_COLS]
    if any(c is None for c in resolved):
        print(f"    [경고] {path.name}: 상권 관련 컬럼을 찾지 못해 건너뜀")
        return None
    df = read_csv_kr(path, usecols=resolved)
    return df.rename(columns=dict(zip(resolved, AREA_COLS)))


def find_hwanghak(files, base_cols, label):
    """상권명에 '황학'이 포함된 행을 모든 연도/분기 파일에서 찾아 중복 제거한다."""
    frames = []
    for f in files:
        df = read_area_cols(f, base_cols)
        if df is None:
            continue
        hit = df[df[AREA_NAME_COL].astype(str).str.contains("황학", na=False)]
        if not hit.empty:
            frames.append(hit[AREA_COLS])

    result = (
        pd.concat(frames, ignore_index=True).drop_duplicates()
        if frames else pd.DataFrame(columns=AREA_COLS)
    )
    print(f"[{label}] 상권명에 '황학' 포함 (중복 제거 후 {len(result)}건)")
    return result


hwanghak_sales = find_hwanghak(sales_files, sales_base_cols, "sales")
hwanghak_sales

상권 코드 컬럼      : 상권_코드
상권명 컬럼         : 상권_코드_명
상권 구분 코드 컬럼 : 상권_구분_코드
상권 구분명 컬럼    : 상권_구분_코드_명


[sales] 상권명에 '황학' 포함 (중복 제거 후 4건)


,상권_코드,상권_코드_명,상권_구분_코드,상권_구분_코드_명
0,3110055,황학동벼룩시장,A,골목상권
6,3110057,황학코아루아파트,A,골목상권
16,3130054,황학동주방가구거리상점가,R,전통시장
25,3130055,"황학시장(서울중앙시장, 신중앙시장)",R,전통시장


In [13]:
hwanghak_stores = find_hwanghak(stores_files, stores_base_cols, "stores")
hwanghak_stores

    [참고] stores_2025.csv: '상권_코드' 컬럼명이 없어 동일 위치의 'trdar_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_코드_명' 컬럼명이 없어 동일 위치의 'trdar_cd_nm'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_구분_코드' 컬럼명이 없어 동일 위치의 'trdar_se_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_구분_코드_명' 컬럼명이 없어 동일 위치의 'trdar_se_cd_nm'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)


[stores] 상권명에 '황학' 포함 (중복 제거 후 4건)


,상권_코드,상권_코드_명,상권_구분_코드,상권_구분_코드_명
0,3110055,황학동벼룩시장,A,골목상권
29,3110057,황학코아루아파트,A,골목상권
69,3130054,황학동주방가구거리상점가,R,전통시장
104,3130055,"황학시장(서울중앙시장, 신중앙시장)",R,전통시장


In [14]:
hwanghak_population = find_hwanghak(population_files, population_base_cols, "population")
hwanghak_population

[population] 상권명에 '황학' 포함 (중복 제거 후 4건)


,상권_코드,상권_코드_명,상권_구분_코드,상권_구분_코드_명
0,3130054,황학동주방가구거리상점가,R,전통시장
1,3130055,"황학시장(서울중앙시장, 신중앙시장)",R,전통시장
2,3110055,황학동벼룩시장,A,골목상권
3,3110057,황학코아루아파트,A,골목상권


In [15]:
sales_codes = set(hwanghak_sales[AREA_CODE_COL])
stores_codes = set(hwanghak_stores[AREA_CODE_COL])
population_codes = set(hwanghak_population[AREA_CODE_COL])

print("sales 황학 상권코드     :", sales_codes)
print("stores 황학 상권코드    :", stores_codes)
print("population 황학 상권코드:", population_codes)
print("\n세 데이터 공통 황학 상권코드:", sales_codes & stores_codes & population_codes)

sales 황학 상권코드     : {3110057, 3130055, 3130054, 3110055}
stores 황학 상권코드    : {3110057, 3130055, 3130054, 3110055}
population 황학 상권코드: {3110055, 3110057, 3130054, 3130055}

세 데이터 공통 황학 상권코드: {3110057, 3130055, 3130054, 3110055}


### 8-1. 황학 4개 상권의 분기별 지속 존재 여부

위에서 세 데이터 모두에 공통으로 존재하는 것을 확인한 4개 상권코드가, 2021Q1(20211)~2025Q4(20254) 전체 20개 분기 동안 **각 데이터에 빠짐없이 존재**하는지 확인한다. '상권-업종' 조합이 아니라 '상권' 자체의 존재 여부만 보는 것이므로 기준년분기 + 상권코드 두 컬럼만 사용한다.

In [16]:
HWANGHAK_CODES = sorted(sales_codes & stores_codes & population_codes)
print("황학 관련 공통 상권코드:", HWANGHAK_CODES)


def collect_key_df(files, base_cols, canonical_cols):
    """여러 파일에서 canonical_cols에 해당하는 컬럼만 찾아 읽고, 표준 컬럼명으로 통일해 이어붙인다.
    컬럼명이 다른 파일(예: stores_2025.csv)도 resolve_column으로 대응 컬럼을 찾아 비교용 임시 결과에서만
    이름을 맞추며, 원본 DataFrame이나 파일은 변경하지 않는다."""
    frames = []
    for f in files:
        resolved = [resolve_column(f, base_cols, c) for c in canonical_cols]
        if any(c is None for c in resolved):
            print(f"    [경고] {f.name}: 필요한 컬럼을 찾지 못해 건너뜀")
            continue
        df = read_csv_kr(f, usecols=resolved)
        frames.append(df.rename(columns=dict(zip(resolved, canonical_cols)))[canonical_cols])
    return pd.concat(frames, ignore_index=True)


EXPECTED_QUARTER_SET = set(expected_quarters)
name_lookup = dict(zip(hwanghak_sales[AREA_CODE_COL], hwanghak_sales[AREA_NAME_COL]))

dataset_specs = [
    ("sales", sales_files, sales_base_cols),
    ("stores", stores_files, stores_base_cols),
    ("population", population_files, population_base_cols),
]

coverage_rows = []
for label, files, base_cols in dataset_specs:
    qc_df = collect_key_df(files, base_cols, [QUARTER_COL, AREA_CODE_COL])
    qc_df = qc_df[qc_df[AREA_CODE_COL].isin(HWANGHAK_CODES)]
    for code in HWANGHAK_CODES:
        present_q = set(qc_df.loc[qc_df[AREA_CODE_COL] == code, QUARTER_COL].unique().tolist())
        missing_q = sorted(EXPECTED_QUARTER_SET - present_q)
        coverage_rows.append({
            "상권코드": code,
            "상권명": name_lookup.get(code, "-"),
            "데이터": label,
            "존재 분기 수": len(present_q),
            "기대 분기 수": len(EXPECTED_QUARTER_SET),
            "누락 분기": "누락 없음" if not missing_q else ", ".join(str(q) for q in missing_q),
            "전체 기간 존재 여부": "YES" if not missing_q else "NO",
        })

hwanghak_quarter_coverage = pd.DataFrame(coverage_rows)
hwanghak_quarter_coverage

황학 관련 공통 상권코드: [3110055, 3110057, 3130054, 3130055]


    [참고] stores_2025.csv: '기준_년분기_코드' 컬럼명이 없어 동일 위치의 'stdr_yyqu_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_코드' 컬럼명이 없어 동일 위치의 'trdar_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)


,상권코드,상권명,데이터,존재 분기 수,기대 분기 수,누락 분기,전체 기간 존재 여부
0,3110055,황학동벼룩시장,sales,20,20,누락 없음,YES
1,3110057,황학코아루아파트,sales,20,20,누락 없음,YES
2,3130054,황학동주방가구거리상점가,sales,20,20,누락 없음,YES
3,3130055,"황학시장(서울중앙시장, 신중앙시장)",sales,20,20,누락 없음,YES
4,3110055,황학동벼룩시장,stores,20,20,누락 없음,YES
5,3110057,황학코아루아파트,stores,20,20,누락 없음,YES
6,3130054,황학동주방가구거리상점가,stores,20,20,누락 없음,YES
7,3130055,"황학시장(서울중앙시장, 신중앙시장)",stores,20,20,누락 없음,YES
8,3110055,황학동벼룩시장,population,20,20,누락 없음,YES
9,3110057,황학코아루아파트,population,20,20,누락 없음,YES


위 표 기준으로, 황학 관련 4개 상권(3110055, 3110057, 3130054, 3130055) 모두 `sales`, `stores`, `population` 세 데이터에서 "존재 분기 수 = 20 = 기대 분기 수", "누락 분기 = 누락 없음", "전체 기간 존재 여부 = YES"로 나타났다. 즉 **4개 상권 모두 2021Q1~2025Q4 전체 20개 분기에 걸쳐 연속적으로 존재하며, 세 데이터 모두에서 시계열 분석이 가능하다.**

## 9. 데이터 간 결합 가능성 확인

실제 merge는 아직 수행하지 않는다. 대신 각 데이터에서 제안된 key 조합으로 `duplicated()`를 확인하여, 해당 key가 실제로 관측 단위(grain)를 유일하게 식별하는지 검증한다.

- `sales`, `stores`: 기준년분기 + 상권코드 + 서비스업종
- `population`: 기준년분기 + 상권코드
- `commercial_area`: 상권코드 (섹션 10에서 GIS 데이터 확인 시 함께 검증)

In [17]:
SERVICE_CODE_COL = [c for c in sales_base_cols if all(k in c for k in ["서비스", "업종", "코드"]) and "명" not in c][0]
print("서비스 업종 코드 컬럼:", SERVICE_CODE_COL)


def check_grain(files, base_cols, key_cols, label):
    """제안된 key로 duplicated()를 확인하여 각 파일의 grain을 검증한다."""
    total_rows, total_dupe = 0, 0
    for f in files:
        resolved = [resolve_column(f, base_cols, c) for c in key_cols]
        if any(c is None for c in resolved):
            print(f"  - {f.name}: key 컬럼을 찾지 못해 건너뜀")
            continue
        df = read_csv_kr(f, usecols=resolved)
        dupe = int(df.duplicated(subset=resolved).sum())
        total_rows += len(df)
        total_dupe += dupe
        status = "중복 없음" if dupe == 0 else f"중복 {dupe}건"
        print(f"  - {f.name}: {len(df)}행, {status}")
    print(f"  -> {label} 전체 {total_rows}행 중 key 중복 {total_dupe}건\n")


sales_key = [QUARTER_COL, AREA_CODE_COL, SERVICE_CODE_COL]
print(f"[sales] key = 기준년분기 + 상권코드 + 서비스업종 ({sales_key})")
check_grain(sales_files, sales_base_cols, sales_key, "sales")

stores_key = [QUARTER_COL, AREA_CODE_COL, SERVICE_CODE_COL]
print(f"[stores] key = 기준년분기 + 상권코드 + 서비스업종 ({stores_key})")
check_grain(stores_files, stores_base_cols, stores_key, "stores")

population_key = [QUARTER_COL, AREA_CODE_COL]
print(f"[population] key = 기준년분기 + 상권코드 ({population_key})")
check_grain(population_files, population_base_cols, population_key, "population")

서비스 업종 코드 컬럼: 서비스_업종_코드
[sales] key = 기준년분기 + 상권코드 + 서비스업종 (['기준_년분기_코드', '상권_코드', '서비스_업종_코드'])


  - sales_2021.csv: 89150행, 중복 없음


  - sales_2022.csv: 88834행, 중복 없음


  - sales_2023.csv: 88246행, 중복 없음


  - sales_2024.csv: 87179행, 중복 없음


...

  - stores_2022.csv: 305587행, 중복 없음


  - stores_2023.csv: 307741행, 중복 없음


  - stores_2024.csv: 306889행, 중복 없음
    [참고] stores_2025.csv: '기준_년분기_코드' 컬럼명이 없어 동일 위치의 'stdr_yyqu_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_코드' 컬럼명이 없어 동일 위치의 'trdar_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '서비스_업종_코드' 컬럼명이 없어 동일 위치의 'svc_induty_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
  - stores_2025.csv: 304775행, 중복 없음
  -> stores 전체 1528872행 중 key 중복 0건

[population] key = 기준년분기 + 상권코드 (['기준_년분기_코드', '상권_코드'])
  - population_20211.csv: 1650행, 중복 없음
  - population_20212.csv: 1650행, 중복 없음
  - population_20213.csv: 1650행, 중복 없음


  - population_20214.csv: 1650행, 중복 없음
  - population_20221.csv: 1650행, 중복 없음
  - population_20222.csv: 1650행, 중복 없음
  - population_20223.csv: 1650행, 중복 없음
  - population_20224.csv: 1649행, 중복 없음
  - population_20231.csv: 1649행, 중복 없음
  - population_20232.csv: 1649행, 중복 없음
  - population_20233.csv: 1649행, 중복 없음
  - population_20234.csv: 1648행, 중복 없음
  - population_20241.csv: 1649행, 중복 없음
  - population_20242.csv: 1649행, 중복 없음


  - population_20243.csv: 1648행, 중복 없음
  - population_20244.csv: 1649행, 중복 없음
  - population_20251.csv: 1650행, 중복 없음
  - population_20252.csv: 1649행, 중복 없음
  - population_20253.csv: 1648행, 중복 없음
  - population_20254.csv: 1648행, 중복 없음
  -> population 전체 32984행 중 key 중복 0건



### 9-1. 데이터 간 key 매칭률 확인 (merge 전 사전 점검)

지금까지는 각 데이터 내부에서 key 중복 여부만 확인했다. 이번에는 실제 merge를 수행하기 전에, 데이터 간 key가 서로 얼마나 겹치는지 집합(set) 비교로 확인하여 merge 시 발생할 수 있는 누락 위험을 사전에 점검한다.

아래 비교는 실제 merge가 아니며, 비교 목적의 임시 key 집합만 생성한다. `stores_2025.csv`처럼 컬럼명이 다른 파일도 앞서 정의한 `resolve_column`으로 대응 컬럼을 찾아 비교용 컬럼명만 통일할 뿐, 원본 DataFrame이나 파일은 변경하지 않는다.

#### 9-1-1. sales ↔ stores (기준년분기 + 상권코드 + 서비스업종)

In [18]:
sales_full_key_df = collect_key_df(sales_files, sales_base_cols, [QUARTER_COL, AREA_CODE_COL, SERVICE_CODE_COL])
stores_full_key_df = collect_key_df(stores_files, stores_base_cols, [QUARTER_COL, AREA_CODE_COL, SERVICE_CODE_COL])

sales_full_keys = set(map(tuple, sales_full_key_df.itertuples(index=False, name=None)))
stores_full_keys = set(map(tuple, stores_full_key_df.itertuples(index=False, name=None)))


def print_key_match(keys_a, keys_b, name_a, name_b):
    """두 key 집합 간 공통/차집합 개수와 매칭률(%)을 출력한다."""
    common = keys_a & keys_b
    only_a = keys_a - keys_b
    only_b = keys_b - keys_a
    print(f"{name_a} unique key 수       : {len(keys_a)}")
    print(f"{name_b} unique key 수       : {len(keys_b)}")
    print(f"양쪽 공통 key 수            : {len(common)}")
    print(f"{name_a}에만 존재하는 key 수  : {len(only_a)}")
    print(f"{name_b}에만 존재하는 key 수  : {len(only_b)}")
    print(f"{name_a} 기준 매칭률(%)      : {len(common) / len(keys_a) * 100:.2f}" if keys_a else f"{name_a} 기준 매칭률(%)      : N/A")
    print(f"{name_b} 기준 매칭률(%)      : {len(common) / len(keys_b) * 100:.2f}" if keys_b else f"{name_b} 기준 매칭률(%)      : N/A")


print("[전체 데이터]")
print_key_match(sales_full_keys, stores_full_keys, "sales", "stores")

hwanghak_sales_full_keys = {k for k in sales_full_keys if k[1] in HWANGHAK_CODES}
hwanghak_stores_full_keys = {k for k in stores_full_keys if k[1] in HWANGHAK_CODES}

print("\n[황학 4개 상권 한정]")
print_key_match(hwanghak_sales_full_keys, hwanghak_stores_full_keys, "sales", "stores")

    [참고] stores_2025.csv: '기준_년분기_코드' 컬럼명이 없어 동일 위치의 'stdr_yyqu_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '상권_코드' 컬럼명이 없어 동일 위치의 'trdar_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)
    [참고] stores_2025.csv: '서비스_업종_코드' 컬럼명이 없어 동일 위치의 'svc_induty_cd'을(를) 대신 사용함 (컬럼명 표기 체계가 다름)


[전체 데이터]
sales unique key 수       : 439141
stores unique key 수       : 1528872
양쪽 공통 key 수            : 439141
sales에만 존재하는 key 수  : 0
stores에만 존재하는 key 수  : 1089731
sales 기준 매칭률(%)      : 100.00
stores 기준 매칭률(%)      : 28.72



[황학 4개 상권 한정]
sales unique key 수       : 717
stores unique key 수       : 3000
양쪽 공통 key 수            : 717
sales에만 존재하는 key 수  : 0
stores에만 존재하는 key 수  : 2283
sales 기준 매칭률(%)      : 100.00
stores 기준 매칭률(%)      : 23.90


#### 9-1-2. sales/stores ↔ population (기준년분기 + 상권코드)

`population`은 업종 단위가 없으므로, `sales`/`stores`에서 먼저 `기준년분기 + 상권코드` 수준으로 unique key를 추출한 뒤 비교한다.

In [19]:
sales_qc_keys = set(map(tuple, sales_full_key_df[[QUARTER_COL, AREA_CODE_COL]].drop_duplicates().itertuples(index=False, name=None)))
stores_qc_keys = set(map(tuple, stores_full_key_df[[QUARTER_COL, AREA_CODE_COL]].drop_duplicates().itertuples(index=False, name=None)))

population_key_df = collect_key_df(population_files, population_base_cols, [QUARTER_COL, AREA_CODE_COL])
population_keys = set(map(tuple, population_key_df.itertuples(index=False, name=None)))

hwanghak_sales_qc_keys = {k for k in sales_qc_keys if k[1] in HWANGHAK_CODES}
hwanghak_stores_qc_keys = {k for k in stores_qc_keys if k[1] in HWANGHAK_CODES}
hwanghak_population_keys = {k for k in population_keys if k[1] in HWANGHAK_CODES}


def match_stats(keys_ref, keys_pop):
    """기준 데이터(sales/stores)와 population 간 key 매칭 통계를 계산한다."""
    common = keys_ref & keys_pop
    return {
        "기준 unique key 수": len(keys_ref),
        "population unique key 수": len(keys_pop),
        "공통 key 수": len(common),
        "기준에만 존재": len(keys_ref - keys_pop),
        "population에만 존재": len(keys_pop - keys_ref),
        "기준 매칭률(%)": round(len(common) / len(keys_ref) * 100, 2) if keys_ref else None,
        "population 매칭률(%)": round(len(common) / len(keys_pop) * 100, 2) if keys_pop else None,
    }


population_match_rows = {
    "sales vs population (전체)": match_stats(sales_qc_keys, population_keys),
    "sales vs population (황학 4개 상권)": match_stats(hwanghak_sales_qc_keys, hwanghak_population_keys),
    "stores vs population (전체)": match_stats(stores_qc_keys, population_keys),
    "stores vs population (황학 4개 상권)": match_stats(hwanghak_stores_qc_keys, hwanghak_population_keys),
}

population_match_df = pd.DataFrame(population_match_rows).T
population_match_df

,기준 unique key 수,population unique key 수,공통 key 수,기준에만 존재,population에만 존재,기준 매칭률(%),population 매칭률(%)
sales vs population (전체),31415.0,32984.0,31399.0,16.0,1585.0,99.95,95.19
sales vs population (황학 4개 상권),80.0,80.0,80.0,0.0,0.0,100.00,100.00
stores vs population (전체),33000.0,32984.0,32984.0,16.0,0.0,99.95,100.00
stores vs population (황학 4개 상권),80.0,80.0,80.0,0.0,0.0,100.00,100.00


## 10. GIS 데이터 기본 구조 확인

`commercial_area.shp`를 `geopandas`로 읽을 수 있는지 확인한다. 환경에 `geopandas`가 설치되어 있지 않다면 환경을 임의로 변경하지 않고, 어떤 패키지가 필요한지만 안내한다.

In [20]:
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
    print("geopandas 사용 가능 (버전:", gpd.__version__, ")")
except ImportError:
    HAS_GEOPANDAS = False
    print("geopandas가 현재 환경에 설치되어 있지 않습니다.")
    print("GIS 데이터를 읽으려면 다음 패키지 설치가 필요합니다: geopandas (의존 패키지: shapely, pyogrio 또는 fiona)")

geopandas가 현재 환경에 설치되어 있지 않습니다.
GIS 데이터를 읽으려면 다음 패키지 설치가 필요합니다: geopandas (의존 패키지: shapely, pyogrio 또는 fiona)


In [21]:
# geopandas 없이도 확인 가능한 보조 파일(.prj, .cpg)은 미리 확인해둔다.
prj_path = COMMERCIAL_AREA_DIR / "commercial_area.prj"
cpg_path = COMMERCIAL_AREA_DIR / "commercial_area.cpg"

print("[.prj] 좌표계(CRS) 정의:")
print(" ", prj_path.read_text(encoding="utf-8"))

print("\n[.cpg] 속성 테이블(dbf) 인코딩:")
print(" ", cpg_path.read_text(encoding="utf-8"))

[.prj] 좌표계(CRS) 정의:
  PROJCS["Korea_2000_Korea_Central_Belt",GEOGCS["GCS_Korea_2000",DATUM["D_Korea_2000",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",200000.0],PARAMETER["False_Northing",500000.0],PARAMETER["Central_Meridian",127.0],PARAMETER["Scale_Factor",1.0],PARAMETER["Latitude_Of_Origin",38.0],UNIT["Meter",1.0]]

[.cpg] 속성 테이블(dbf) 인코딩:
  UTF-8


In [22]:
if HAS_GEOPANDAS:
    shp_path = COMMERCIAL_AREA_DIR / "commercial_area.shp"
    gdf = gpd.read_file(shp_path)

    print("shape:", gdf.shape)
    print("CRS:", gdf.crs)
    print("컬럼명:", gdf.columns.tolist())
    print("geometry type:", gdf.geom_type.unique().tolist())
    display(gdf.head())
else:
    gdf = None
    print("geopandas 미설치로 GIS 데이터 구조 확인을 건너뜁니다.")

geopandas 미설치로 GIS 데이터 구조 확인을 건너뜁니다.


In [23]:
if HAS_GEOPANDAS:
    # Shapefile 컬럼명은 표준상 10자 이내로 축약되는 경우가 많으므로,
    # 앞서 사용한 것과 동일한 한글 컬럼명이 아닐 수 있다. 실제 컬럼명 목록을 그대로 출력해서 확인한다.
    print("GIS 데이터 컬럼명 목록:", gdf.columns.tolist())

    code_like_cols = [c for c in gdf.columns if ("상권" in c and "코드" in c) or "TRDAR" in c.upper() or c.upper() in ("CODE", "AREA_CD")]
    name_like_cols = [c for c in gdf.columns if ("상권" in c and "명" in c) or "NM" in c.upper() or "NAME" in c.upper()]
    print("상권 코드로 추정되는 컬럼:", code_like_cols)
    print("상권명으로 추정되는 컬럼 :", name_like_cols)

    if name_like_cols:
        target_col = name_like_cols[0]
        hwanghak_gis = gdf[gdf[target_col].astype(str).str.contains("황학", na=False)]
        print(f"\n'{target_col}' 컬럼 기준 상권명에 '황학' 포함된 영역: {len(hwanghak_gis)}건")
        display(hwanghak_gis[code_like_cols + name_like_cols] if code_like_cols else hwanghak_gis[name_like_cols])
    else:
        print("\n상권명으로 추정되는 컬럼을 찾지 못했습니다. 위 컬럼명 목록을 직접 확인해야 합니다.")
else:
    print("geopandas 미설치로 GIS 데이터 내 황학 상권 탐색을 건너뜁니다.")

geopandas 미설치로 GIS 데이터 내 황학 상권 탐색을 건너뜁니다.


## 11. 최종 데이터 점검 요약

### 데이터 구조 요약

| 데이터 | 기간 | 파일 수 | 공간 단위 | 업종 단위 | 시간대 정보 | 연령 정보 | 주요 결합키 |
|---|---|---:|---|---|---|---|---|
| sales | 2021Q1~2025Q4 | 5 (연도별, 총 439,141행) | 상권 | 서비스 업종별 | 있음 (6구간 + 요일별) | 있음 (10~60대 이상 + 성별) | 기준년분기 + 상권코드 + 서비스업종 |
| stores | 2021Q1~2025Q4 | 5 (연도별, 총 1,528,872행) | 상권 | 서비스 업종별 | 없음 | 없음 | 기준년분기 + 상권코드 + 서비스업종 |
| population | 2021Q1~2025Q4 | 20 (분기별, 총 32,984행) | 상권 | 없음 (상권 전체 유동인구) | 있음 (6구간 + 요일별) | 있음 (10~60대 이상 + 성별) | 기준년분기 + 상권코드 |
| commercial_area | 시점 정보 없음 (정적 경계) | 1 (shapefile, 부속 4파일) | 상권(폴리곤) | 없음 | 없음 | 없음 | 상권코드 (컬럼 존재 여부 확인 필요) |

### 체크리스트

1. **황학 관련 상권 데이터가 존재하는가?** → **YES**. `sales`, `stores`, `population` 세 데이터 모두에서 상권명에 '황학'이 포함된 4개 상권(황학동벼룩시장·황학코아루아파트 = 골목상권(A), 황학동주방가구거리상점가·황학시장(서울중앙시장, 신중앙시장) = 전통시장(R))이 동일한 상권코드로 공통 확인됨.
2. **2021Q1~2025Q4 시계열이 확보되는가?** → **YES**. `sales`, `stores`, `population` 모두 20211~20254 전체 20개 분기가 누락 없이 포함됨. 섹션 8-1에서 황학 4개 상권 각각에 대해서도 분기별 존재 여부를 개별 확인함(결과는 섹션 8-1 참고).
3. **업종별 매출 분석이 가능한가?** → **YES**. `sales`에 `서비스_업종_코드`/`서비스_업종_코드_명` 컬럼 존재.
4. **시간대별 매출 분석이 가능한가?** → **YES**. `sales`에 시간대_00~06 등 6개 구간별 매출 금액/건수 컬럼 존재.
5. **연령대별 매출 분석이 가능한가?** → **YES**. `sales`에 연령대_10~60대 이상, 성별 매출 금액/건수 컬럼 존재.
6. **점포 데이터와 결합 가능한가?** → **YES (단, 전처리 필요)**. `기준년분기 + 상권코드 + 서비스업종` key로 `sales`, `stores` 모두 grain 중복이 없음을 확인했고, 섹션 9-1에서 두 데이터 간 key 매칭률도 실제로 확인함. 다만 `stores_2025.csv`는 다른 연도와 달리 컬럼명이 영문 코드 체계(`stdr_yyqu_cd` 등)로 되어 있어, 실제 merge 전에는 컬럼명 매핑(정규화) 작업이 필요함.
7. **길단위인구 데이터와 결합 가능한가?** → **YES**. `기준년분기 + 상권코드` key로 grain 중복이 없음을 확인했고, 섹션 9-1에서 `sales`/`stores`와 `population` 간 key 매칭률을 실제로 계산하여 결합 가능성을 확인함.
8. **GIS 영역 데이터와 상권코드로 연결 가능한가?** → **확인 필요**. 현재 실행 환경에 `geopandas`가 설치되어 있지 않아 shapefile의 실제 컬럼명과 상권코드 존재 여부를 확인하지 못했다. `.prj` 파일을 통해 좌표계가 `Korea_2000_Korea_Central_Belt`(중부원점)임은 확인했다. `geopandas`(및 `shapely`, `pyogrio`/`fiona`) 설치 후 재확인이 필요하다.

### 종합 판단

세 개의 표 형태 데이터(`sales`, `stores`, `population`)는 2021Q1~2025Q4의 전체 20개 분기가 누락 없이 확보되어 있고, 상권코드를 공통 key로 결합 가능함이 확인되었다. '황학' 관련 상권을 탐색한 결과 전통시장(R) 성격의 상권(황학시장, 주방가구거리상점가)과 골목상권(A) 성격의 상권(황학동벼룩시장, 황학코아루아파트)이 모두 확인되어, "전환 중인 상권"이라는 프로젝트 가설을 검증하기 위한 비교 대상과 데이터 구조가 확보되었다고 판단된다. 실제 상권 전환 여부(매출 증감, 업종 변화, 시간대별 소비 패턴 변화 등)는 이 노트북에서 다루지 않았으며, 이후 시계열 EDA를 통해 별도로 검증해야 한다. 다만 `stores_2025.csv`의 컬럼명 체계 불일치는 이후 전처리 단계에서 반드시 정규화가 필요하며, GIS 경계 데이터와의 실제 연결 가능성은 `geopandas` 설치 후 별도로 확인해야 한다. 또한 이 단계에서 확인된 '황학' 4개 상권이 실제 황학동 행정/계획구역과 정확히 일치하는지는 GIS 경계 데이터 확인 후 추가로 검증이 필요하며, 현재 결과만으로 분석 대상 상권 범위를 확정할 수는 없다. 종합적으로, 결측치 처리·컬럼명 정규화·merge 등 다음 단계 전처리를 진행할 수 있는 구조적 기반은 갖춰져 있다고 판단된다.